# Subject 01 Visual Brain MindEye Low-Level SD-VAE


# 1. Cache + Train MindEye Low-Level SD-VAE Model
Encode NSD images with `stabilityai/sd-vae-ft-mse`, cache latents once, train a MindEye-1-style residual MLP plus CNN upsampler from fMRI to SD-VAE latents, decode shared-1000 predictions, and evaluate reconstructions.


## Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, glob, json, random, shutil, time, gc, sys, subprocess
from collections import defaultdict, OrderedDict

import importlib.util
_missing = []
for module_name, package_name in [("diffusers", "diffusers"), ("accelerate", "accelerate"), ("transformers", "transformers"), ("safetensors", "safetensors"), ("skimage", "scikit-image")]:
    if importlib.util.find_spec(module_name) is None:
        _missing.append(package_name)
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from diffusers import AutoencoderKL
try:
    from diffusers.models.autoencoders.vae import Decoder
except ImportError:
    from diffusers.models.vae import Decoder

BASE = "/content"
DRIVE_PROJECT_DIR = f"{BASE}/drive/Shareddrives/FMRI_Paper"
INPUT_DIR = f"{DRIVE_PROJECT_DIR}/inputs"
BETA_DIR = f"{BASE}/subject01_visual_brain_responses"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/outputs/subject01_mindeye_lowlevel_sdvae_{time.strftime('%Y%m%d_%H%M%S')}"
SUBJECT = "subj01"

SEED = 0
VAL_FRAC = 0.05
BATCH_SIZE = 256
LOWLEVEL_EPOCHS = 20
LOWLEVEL_LR = 1e-4
LOWLEVEL_HIDDEN_DIM = 4096
LOWLEVEL_RES_BLOCKS = 4
LOWLEVEL_BOTTLENECK_SHAPE = (64, 16, 16)
LOWLEVEL_INPUT_DROPOUT = 0.50
LOWLEVEL_RES_DROPOUT = 0.25
LOWLEVEL_CONT_WEIGHT = 0.0
VAE_MODEL_ID = "stabilityai/sd-vae-ft-mse"
VAE_IMAGE_SIZE = 512
VAE_ENCODE_BATCH = 12
VAE_DECODE_BATCH = 32
VAE_CACHE_SHARD_SIZE = 128
SDVAE_CACHE_DIR = f"{INPUT_DIR}/subject01_sdvae_ft_mse_features_{VAE_IMAGE_SIZE}"
SDVAE_LOCAL_CACHE_DIR = f"{BASE}/subject01_sdvae_ft_mse_features_{VAE_IMAGE_SIZE}"
RUN_SD_VAE_CACHE_BUILDER = not os.path.exists(f"{SDVAE_CACHE_DIR}/metadata.json")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
os.makedirs(OUTPUT_DIR, exist_ok=True)

for name in ["subject01_visual_brain_responses"]:
    dst = f"{BASE}/{name}"
    if not os.path.exists(dst):
        shutil.copytree(f"{INPUT_DIR}/{name}", dst)

print(f"device={DEVICE} batch_size={BATCH_SIZE} amp={USE_AMP} output={OUTPUT_DIR}")


## Load Data


In [ ]:
import urllib.request
from scipy.io import loadmat

def session_id(path):
    return int(re.findall(r"\d+", os.path.basename(path))[-1])

beta_sess = {session_id(p): p for p in glob.glob(f"{BETA_DIR}/{SUBJECT}_visualroi_session*.pt")}
sessions = sorted(beta_sess)
assert sessions, "No fMRI sessions found."

betas = torch.cat([torch.load(beta_sess[s], map_location="cpu").float() for s in tqdm(sessions, desc="load fMRI")])

EXP = f"{BASE}/nsd_expdesign.mat"
if not os.path.exists(EXP):
    urllib.request.urlretrieve("https://natural-scenes-dataset.s3.amazonaws.com/nsddata/experiments/nsd/nsd_expdesign.mat", EXP)

mat = loadmat(EXP)
masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1
subjectim = mat["subjectim"].astype(np.int64) - 1
imgbrick_ids = subjectim[int(SUBJECT[-2:]) - 1, masterordering]
shared_ids = set(mat["sharedix"].reshape(-1).astype(np.int64) - 1)

def image_id_for_concat_trial(gidx):
    session = sessions[int(gidx) // 750]
    offset = int(gidx) % 750
    return int(imgbrick_ids[(session - 1) * 750 + offset])

img_of = np.array([image_id_for_concat_trial(gidx) for gidx in range(len(betas))], dtype=np.int64)
is_shared = np.array([int(img_id) in shared_ids for img_id in img_of])

nonshared_img_ids = np.array(sorted(set(img_of[~is_shared])), dtype=np.int64)
image_perm = np.random.RandomState(SEED).permutation(nonshared_img_ids)
n_val_images = int(VAL_FRAC * len(image_perm))
val_img_ids = set(map(int, image_perm[:n_val_images]))
train_img_ids = set(map(int, image_perm[n_val_images:]))
in_val = np.array([int(img_id) in val_img_ids for img_id in img_of])
train_idx = torch.from_numpy(np.where(~is_shared & ~in_val)[0]).long()
val_idx = torch.from_numpy(np.where(~is_shared & in_val)[0]).long()

shared_groups = defaultdict(list)
for gidx, img_id in enumerate(img_of):
    if int(img_id) in shared_ids:
        shared_groups[int(img_id)].append(gidx)
shared_img_ids = list(shared_groups)
shared_eval_betas = torch.stack([betas[idxs].mean(0) for idxs in shared_groups.values()])

beta_mean = betas[train_idx].mean(0)
beta_std = betas[train_idx].std(0) + 1e-6
x = (betas - beta_mean) / beta_std
shared_eval_x = (shared_eval_betas - beta_mean) / beta_std
INPUT_DIM = x.shape[1]
unique_subject_img_ids = np.array(sorted(set(img_of.tolist())), dtype=np.int64)

def average_trials_by_image(indices):
    groups = defaultdict(list)
    for gidx in indices.tolist():
        groups[int(img_of[gidx])].append(gidx)
    ids = sorted(groups)
    avg_betas = torch.stack([betas[groups[img_id]].mean(0) for img_id in ids])
    avg_x = (avg_betas - beta_mean) / beta_std
    return avg_x, torch.tensor(ids, dtype=torch.long)

train_x, train_img_id_tensor = average_trials_by_image(train_idx)
val_x, val_img_id_tensor = average_trials_by_image(val_idx)
test_x = shared_eval_x
test_img_id_tensor = torch.tensor(shared_img_ids, dtype=torch.long)

print(f"sessions={sessions}")
print(f"betas={tuple(betas.shape)} input_dim={INPUT_DIM}")
print(f"unique_subject_images={len(unique_subject_img_ids)}")
print(f"train_trials={len(train_idx)} train_images={len(train_x)} val_trials={len(val_idx)} val_images={len(val_x)} shared_test_images={len(shared_img_ids)}")


## Load SD-VAE


In [ ]:
vae = AutoencoderKL.from_pretrained(VAE_MODEL_ID).to(DEVICE).eval()
vae.requires_grad_(False)
VAE_SCALING_FACTOR = float(getattr(vae.config, "scaling_factor", 0.18215))
print(f"loaded {VAE_MODEL_ID} scaling_factor={VAE_SCALING_FACTOR}")


## Temporary Cache Builder
Run once. It opens `nsd_stimuli.hdf5`, encodes all subject01 images into SD-VAE latents, and saves resized ground-truth images for later eval.


In [ ]:
def copy_with_progress(src, dst, desc, chunk_mb=128):
    total = os.path.getsize(src)
    tmp = f"{dst}.part"
    if os.path.exists(tmp):
        os.remove(tmp)
    with open(src, "rb") as fsrc, open(tmp, "wb") as fdst, tqdm(total=total, unit="B", unit_scale=True, desc=desc) as pbar:
        while True:
            chunk = fsrc.read(chunk_mb * 1024 * 1024)
            if not chunk:
                break
            fdst.write(chunk)
            pbar.update(len(chunk))
    if os.path.getsize(tmp) != total:
        raise IOError(f"Incomplete copy: {tmp}")
    shutil.copystat(src, tmp)
    os.replace(tmp, dst)

def find_stimulus_dataset(h5):
    for key in ["imgBrick", "images", "stimuli"]:
        if key in h5:
            return h5[key]
    keys = list(h5.keys())
    raise KeyError(f"Could not find NSD image dataset. HDF5 keys={keys}")

def preprocess_nsd_uint8(images_np):
    img = torch.from_numpy(images_np).permute(0, 3, 1, 2).float().div(255.0)
    img = F.interpolate(img, size=(VAE_IMAGE_SIZE, VAE_IMAGE_SIZE), mode="bicubic", align_corners=False).clamp(0, 1)
    return img

@torch.no_grad()
def encode_sdvae_images(img01):
    img_m11 = img01.to(DEVICE, non_blocking=True).mul(2).sub(1)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        posterior = vae.encode(img_m11).latent_dist
        latents = posterior.mode() * VAE_SCALING_FACTOR
    return latents.float().cpu()

def build_sdvae_cache():
    os.makedirs(SDVAE_CACHE_DIR, exist_ok=True)
    stim_path = f"{BASE}/nsd_stimuli.hdf5"
    src_stim_path = f"{INPUT_DIR}/nsd_stimuli.hdf5"
    if not os.path.exists(stim_path):
        if not os.path.exists(src_stim_path):
            raise FileNotFoundError(f"Missing NSD stimuli HDF5: {src_stim_path}")
        copy_with_progress(src_stim_path, stim_path, "copy nsd_stimuli.hdf5")

    shards = []
    with h5py.File(stim_path, "r") as h5:
        img_ds = find_stimulus_dataset(h5)
        for shard_i, start in enumerate(tqdm(range(0, len(unique_subject_img_ids), VAE_CACHE_SHARD_SIZE), desc="SD-VAE target shards")):
            ids = unique_subject_img_ids[start:start + VAE_CACHE_SHARD_SIZE]
            latents, images_u8 = [], []
            for b0 in range(0, len(ids), VAE_ENCODE_BATCH):
                batch_ids = ids[b0:b0 + VAE_ENCODE_BATCH]
                images_np = img_ds[batch_ids]
                img01 = preprocess_nsd_uint8(images_np)
                latents.append(encode_sdvae_images(img01).half())
                images_u8.append(img01.mul(255).round().byte())
            shard = {
                "img_ids": ids.astype(int).tolist(),
                "latents": torch.cat(latents, dim=0),
                "images_u8": torch.cat(images_u8, dim=0),
            }
            shard_file = f"sdvae_features_{shard_i:05d}.pt"
            torch.save(shard, f"{SDVAE_CACHE_DIR}/{shard_file}")
            shards.append({"file": shard_file, "img_ids": shard["img_ids"]})
            del shard, latents, images_u8
            gc.collect()

    sample = torch.load(f"{SDVAE_CACHE_DIR}/{shards[0]['file']}", map_location="cpu")
    metadata = {
        "model_id": VAE_MODEL_ID,
        "image_size": VAE_IMAGE_SIZE,
        "scaling_factor": VAE_SCALING_FACTOR,
        "latent_shape": list(sample["latents"].shape[1:]),
        "num_images": int(len(unique_subject_img_ids)),
        "dtype": "float16",
        "shard_size": VAE_CACHE_SHARD_SIZE,
        "shards": shards,
    }
    with open(f"{SDVAE_CACHE_DIR}/metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"saved metadata={SDVAE_CACHE_DIR}/metadata.json")

if RUN_SD_VAE_CACHE_BUILDER:
    build_sdvae_cache()
else:
    print(f"cache exists, skipping builder: {SDVAE_CACHE_DIR}")

if os.path.exists(f"{SDVAE_CACHE_DIR}/metadata.json") and not os.path.exists(f"{SDVAE_LOCAL_CACHE_DIR}/metadata.json"):
    shutil.copytree(SDVAE_CACHE_DIR, SDVAE_LOCAL_CACHE_DIR)
    print(f"copied SD-VAE cache to local disk: {SDVAE_LOCAL_CACHE_DIR}")


## Load Cached SD-VAE Targets


In [ ]:
SDVAE_READ_CACHE_DIR = SDVAE_LOCAL_CACHE_DIR if os.path.exists(f"{SDVAE_LOCAL_CACHE_DIR}/metadata.json") else SDVAE_CACHE_DIR
metadata_path = f"{SDVAE_READ_CACHE_DIR}/metadata.json"
if not os.path.exists(metadata_path):
    raise FileNotFoundError(f"Missing SD-VAE cache metadata: {metadata_path}")

with open(metadata_path) as f:
    sdvae_manifest = json.load(f)
LATENT_SHAPE = tuple(sdvae_manifest["latent_shape"])
LATENT_DIM = int(np.prod(LATENT_SHAPE))
print(f"reading SD-VAE targets from {SDVAE_READ_CACHE_DIR}")
print(f"latent_shape={LATENT_SHAPE} latent_dim={LATENT_DIM}")

_sdvae_lookup = {}
for shard in sdvae_manifest["shards"]:
    shard_path = f"{SDVAE_READ_CACHE_DIR}/{shard['file']}"
    if not os.path.exists(shard_path):
        raise FileNotFoundError(f"Missing SD-VAE shard: {shard_path}")
    for row, img_id in enumerate(shard["img_ids"]):
        _sdvae_lookup[int(img_id)] = (shard_path, row)

_sdvae_lru = OrderedDict()
SDVAE_CACHE_MAX_SHARDS_IN_RAM = 32

def _load_sdvae_shard(path):
    if path in _sdvae_lru:
        _sdvae_lru.move_to_end(path)
        return _sdvae_lru[path]
    try:
        shard = torch.load(path, map_location="cpu", mmap=True)
    except TypeError:
        shard = torch.load(path, map_location="cpu")
    _sdvae_lru[path] = shard
    while len(_sdvae_lru) > SDVAE_CACHE_MAX_SHARDS_IN_RAM:
        _sdvae_lru.popitem(last=False)
    return shard

def sdvae_targets_for_ids(img_ids, include_images=False):
    ids = [int(i) for i in img_ids.detach().cpu().tolist()]
    missing = [i for i in ids if i not in _sdvae_lookup]
    if missing:
        raise KeyError(f"Missing cached SD-VAE targets for image ids: {missing[:10]}")
    latents = torch.empty((len(ids), *LATENT_SHAPE), dtype=torch.float16)
    images = torch.empty((len(ids), 3, VAE_IMAGE_SIZE, VAE_IMAGE_SIZE), dtype=torch.uint8) if include_images else None
    by_shard = defaultdict(list)
    for out_row, img_id in enumerate(ids):
        shard_path, shard_row = _sdvae_lookup[img_id]
        by_shard[shard_path].append((out_row, shard_row))
    for shard_path, pairs in by_shard.items():
        shard = _load_sdvae_shard(shard_path)
        out_rows = [p[0] for p in pairs]
        shard_rows = torch.as_tensor([p[1] for p in pairs], dtype=torch.long)
        latents[out_rows] = shard["latents"][shard_rows]
        if include_images:
            images[out_rows] = shard["images_u8"][shard_rows]
    out = {"latents": latents.to(DEVICE, non_blocking=True).float()}
    if include_images:
        out["images"] = images.float().div(255.0)
    return out

class SDVAEShardBatchSampler(torch.utils.data.Sampler):
    def __init__(self, img_ids, batch_size, shuffle=False, seed=0):
        self.img_ids = [int(i) for i in img_ids]
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self.groups = defaultdict(list)
        for row, img_id in enumerate(self.img_ids):
            self.groups[_sdvae_lookup[img_id][0]].append(row)

    def __iter__(self):
        rng = np.random.RandomState(self.seed + self.epoch)
        shard_paths = list(self.groups)
        if self.shuffle:
            rng.shuffle(shard_paths)
        batch = []
        for shard_path in shard_paths:
            rows = np.array(self.groups[shard_path], dtype=np.int64)
            if self.shuffle:
                rng.shuffle(rows)
            for row in rows.tolist():
                batch.append(row)
                if len(batch) == self.batch_size:
                    yield batch
                    batch = []
        if batch:
            yield batch
        self.epoch += 1

    def __len__(self):
        return (len(self.img_ids) + self.batch_size - 1) // self.batch_size

def make_cached_sdvae_loader(x_tensor, img_id_tensor, batch_size, shuffle=False):
    ds = TensorDataset(x_tensor, img_id_tensor)
    sampler = SDVAEShardBatchSampler(img_id_tensor.tolist(), batch_size, shuffle=shuffle, seed=SEED)
    return DataLoader(ds, batch_sampler=sampler, pin_memory=True)

train_loader = make_cached_sdvae_loader(train_x, train_img_id_tensor, BATCH_SIZE, shuffle=True)
train_eval_loader = make_cached_sdvae_loader(train_x, train_img_id_tensor, BATCH_SIZE, shuffle=False)
val_loader = make_cached_sdvae_loader(val_x, val_img_id_tensor, BATCH_SIZE, shuffle=False)
test_loader = make_cached_sdvae_loader(test_x, test_img_id_tensor, BATCH_SIZE, shuffle=False)

sample = sdvae_targets_for_ids(torch.tensor(shared_img_ids[:2]), include_images=True)
print("sample latents", tuple(sample["latents"].shape), "sample images", tuple(sample["images"].shape))
print(f"loaders: train_batches={len(train_loader)} val_batches={len(val_loader)} test_batches={len(test_loader)}")

# MindEye trains directly in scaled SD-VAE latent space; no per-dimension latent whitening.


## MindEye-1 Low-Level Architecture
Residual MLP maps fMRI to a `16x16x64` tensor; a CNN upsampler maps it to the full SD-VAE latent grid.


In [ ]:
class MindEyeResidualBlock(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(h, h, bias=False),
            nn.LayerNorm(h),
            nn.SiLU(inplace=True),
            nn.Dropout(LOWLEVEL_RES_DROPOUT),
        )

    def forward(self, x):
        return self.net(x)

class MindEyeLowLevelSDVAE(nn.Module):
    def __init__(self, input_dim, latent_shape, use_cont=LOWLEVEL_CONT_WEIGHT > 0):
        super().__init__()
        if tuple(latent_shape) != (4, 64, 64):
            raise ValueError(f"MindEye-1 low-level expects SD-VAE latents (4, 64, 64); got {latent_shape}. Use VAE_IMAGE_SIZE=512 and rebuild cache.")
        h = LOWLEVEL_HIDDEN_DIM
        self.use_cont = use_cont
        self.lin0 = nn.Sequential(
            nn.Linear(input_dim, h, bias=False),
            nn.LayerNorm(h),
            nn.SiLU(inplace=True),
            nn.Dropout(LOWLEVEL_INPUT_DROPOUT),
        )
        self.mlp = nn.ModuleList([MindEyeResidualBlock(h) for _ in range(LOWLEVEL_RES_BLOCKS)])
        self.lin1 = nn.Linear(h, int(np.prod(LOWLEVEL_BOTTLENECK_SHAPE)), bias=False)
        self.norm = nn.GroupNorm(1, LOWLEVEL_BOTTLENECK_SHAPE[0])
        self.upsampler = Decoder(
            in_channels=64,
            out_channels=4,
            up_block_types=["UpDecoderBlock2D", "UpDecoderBlock2D", "UpDecoderBlock2D"],
            block_out_channels=[64, 128, 256],
            layers_per_block=1,
        )
        self.maps_projector = nn.Sequential(
            nn.Conv2d(64, 512, 1, bias=False),
            nn.GroupNorm(1, 512),
            nn.ReLU(True),
            nn.Conv2d(512, 512, 1, bias=False),
            nn.GroupNorm(1, 512),
            nn.ReLU(True),
            nn.Conv2d(512, 512, 1, bias=True),
        ) if use_cont else nn.Identity()

    def forward(self, x, return_transformer_feats=False):
        x = self.lin0(x)
        residual = x
        for block in self.mlp:
            x = block(x)
            x = x + residual
            residual = x
        z16 = self.lin1(x).reshape(x.shape[0], *LOWLEVEL_BOTTLENECK_SHAPE).contiguous()
        z16 = self.norm(z16)
        z64 = self.upsampler(z16)
        if return_transformer_feats:
            return z64, self.maps_projector(z16).flatten(2).permute(0, 2, 1)
        return z64

model = MindEyeLowLevelSDVAE(INPUT_DIM, LATENT_SHAPE).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LOWLEVEL_LR, weight_decay=1e-2)
steps_per_epoch = max(len(train_loader), 1)
lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LOWLEVEL_LR,
    total_steps=LOWLEVEL_EPOCHS * steps_per_epoch,
    final_div_factor=1000,
    pct_start=2 / LOWLEVEL_EPOCHS,
)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"parameters={n_params:,} steps_per_epoch={steps_per_epoch}")


## Train MindEye Low-Level Model


In [ ]:
def sdvae_latent_loss(pred_raw, target_raw):
    latent_mae = F.l1_loss(pred_raw, target_raw)
    latent_mse = F.mse_loss(pred_raw, target_raw)
    latent_cos = F.cosine_similarity(pred_raw.flatten(1), target_raw.flatten(1), dim=1).mean()
    loss = latent_mae / VAE_SCALING_FACTOR
    return {"loss": loss, "latent_mae": latent_mae, "latent_mse": latent_mse, "latent_cos": latent_cos}

@torch.no_grad()
def evaluate_lowlevel_model(loader, max_batches=None):
    model.eval()
    totals = defaultdict(float)
    seen = 0
    for batch_i, (xb, img_ids) in enumerate(tqdm(loader, desc="eval lowlevel", leave=False)):
        if max_batches is not None and batch_i >= max_batches:
            break
        xb = xb.to(DEVICE, non_blocking=True)
        target = sdvae_targets_for_ids(img_ids)["latents"]
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            parts = sdvae_latent_loss(model(xb), target)
        n = xb.size(0)
        for k, v in parts.items():
            totals[k] += n * float(v.detach().cpu())
        seen += n
    return {k: v / max(seen, 1) for k, v in totals.items()}

history = []
best_val = float("inf")
for epoch in range(1, LOWLEVEL_EPOCHS + 1):
    model.train()
    totals = defaultdict(float)
    seen = 0
    pbar = tqdm(train_loader, desc=f"mindeye lowlevel {epoch:03d}/{LOWLEVEL_EPOCHS}", leave=False)
    for xb, img_ids in pbar:
        xb = xb.to(DEVICE, non_blocking=True)
        target = sdvae_targets_for_ids(img_ids)["latents"]
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            parts = sdvae_latent_loss(model(xb), target)
            loss = parts["loss"]
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        n = xb.size(0)
        for k, v in parts.items():
            totals[k] += n * float(v.detach().cpu())
        seen += n
        pbar.set_postfix(loss=float(loss.detach().cpu()), mae=float(parts["latent_mae"].detach().cpu()), lr=optimizer.param_groups[0]["lr"])

    row = {f"train_{k}": v / seen for k, v in totals.items()}
    val_scores = evaluate_lowlevel_model(val_loader)
    row.update({f"val_{k}": v for k, v in val_scores.items()})
    row["epoch"] = epoch
    history.append(row)
    tqdm.write(f"epoch={epoch:03d} train_loss={row['train_loss']:.5f} val_loss={row['val_loss']:.5f} val_mae={row['val_latent_mae']:.5f} val_cos={row['val_latent_cos']:.4f}")

    if row["val_loss"] < best_val:
        best_val = row["val_loss"]
        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "lr_scheduler": lr_scheduler.state_dict(),
            "beta_mean": beta_mean,
            "beta_std": beta_std,
            "latent_shape": LATENT_SHAPE,
            "latent_dim": LATENT_DIM,
            "vae_model_id": VAE_MODEL_ID,
            "vae_scaling_factor": VAE_SCALING_FACTOR,
            "vae_image_size": VAE_IMAGE_SIZE,
            "hidden_dim": LOWLEVEL_HIDDEN_DIM,
            "res_blocks": LOWLEVEL_RES_BLOCKS,
            "bottleneck_shape": LOWLEVEL_BOTTLENECK_SHAPE,
            "loss": "MindEye raw SD-VAE latent L1 / scaling_factor",
        }, f"{OUTPUT_DIR}/best_mindeye_lowlevel_sdvae.pt")

with open(f"{OUTPUT_DIR}/mindeye_lowlevel_sdvae_history.json", "w") as f:
    json.dump(history, f, indent=2)
print(f"saved={OUTPUT_DIR}/best_mindeye_lowlevel_sdvae.pt")


## Evaluate MindEye Low-Level Model


In [ ]:
ckpt = torch.load(f"{OUTPUT_DIR}/best_mindeye_lowlevel_sdvae.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
final_train = evaluate_lowlevel_model(train_eval_loader, max_batches=50)
final_val = evaluate_lowlevel_model(val_loader)
final_test = evaluate_lowlevel_model(test_loader)

summary = {"train_sampled": final_train, "val": final_val, "shared1000": final_test}
with open(f"{OUTPUT_DIR}/mindeye_lowlevel_sdvae_eval.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

plt.plot([h["epoch"] for h in history], [h["train_loss"] for h in history], label="train")
plt.plot([h["epoch"] for h in history], [h["val_loss"] for h in history], label="val")
plt.xlabel("epoch")
plt.ylabel("SD-VAE latent L1 / scale")
plt.legend()
plt.savefig(f"{OUTPUT_DIR}/mindeye_lowlevel_sdvae_train_val_loss.png", dpi=180, bbox_inches="tight")
plt.show()


# 2. Decode Predicted SD-VAE Latents


In [ ]:
@torch.no_grad()
def decode_sdvae_latents(latents):
    latents = latents.to(DEVICE, non_blocking=True) / VAE_SCALING_FACTOR
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        imgs = vae.decode(latents).sample
    return imgs.float().add(1).div(2).clamp(0, 1).cpu()

@torch.no_grad()
def predict_sdvae_latents(xb):
    model.eval()
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        return model(xb.to(DEVICE, non_blocking=True)).float()

RECONS_PATH = f"{OUTPUT_DIR}/mindeye_lowlevel_sdvae_shared1000_recons.pt"
IDS_PATH = f"{OUTPUT_DIR}/mindeye_lowlevel_sdvae_shared1000_img_ids.json"

def decode_shared1000():
    if os.path.exists(RECONS_PATH):
        print(f"loading cached reconstructions: {RECONS_PATH}")
        return torch.load(RECONS_PATH, map_location="cpu")
    recons = []
    for xb, img_ids in tqdm(test_loader, desc="decode shared-1000"):
        pred_latents = predict_sdvae_latents(xb)
        for start in range(0, len(pred_latents), VAE_DECODE_BATCH):
            recons.append(decode_sdvae_latents(pred_latents[start:start + VAE_DECODE_BATCH]))
    recons = torch.cat(recons, dim=0).clamp(0, 1)
    torch.save(recons, RECONS_PATH)
    with open(IDS_PATH, "w") as f:
        json.dump([int(i) for i in shared_img_ids[:len(recons)]], f)
    print(f"saved={RECONS_PATH} shape={tuple(recons.shape)}")
    return recons

all_recons = decode_shared1000()
print("all_recons", tuple(all_recons.shape))


# 3. Shared-1000 Reconstruction Eval


In [ ]:
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim_fn

all_recons = torch.load(RECONS_PATH, map_location="cpu").float().clamp(0, 1)
EVAL_N = min(1000, len(all_recons), len(shared_img_ids))
recons_eval = all_recons[:EVAL_N]
ids_eval = torch.tensor(shared_img_ids[:EVAL_N], dtype=torch.long)

@torch.no_grad()
def eval_reconstruction_metrics(recons, img_ids, batch_size=64):
    pixel_mse_total = 0.0
    pixcorr_scores = []
    ssim_scores = []
    latent_mse_total = 0.0
    latent_cos_total = 0.0
    seen = 0
    for start in tqdm(range(0, len(recons), batch_size), desc="eval recons"):
        rec = recons[start:start + batch_size].float()
        batch_ids = img_ids[start:start + batch_size]
        cache = sdvae_targets_for_ids(batch_ids, include_images=True)
        true_img = cache["images"].float()
        n = len(rec)
        pixel_mse_total += n * float(F.mse_loss(rec, true_img).cpu())
        r = rec.flatten(1).numpy()
        t = true_img.flatten(1).numpy()
        pixcorr_scores.extend(float(np.corrcoef(t[i], r[i])[0, 1]) for i in range(n))
        rec_gray = rgb2gray(rec.permute(0, 2, 3, 1).numpy())
        true_gray = rgb2gray(true_img.permute(0, 2, 3, 1).numpy())
        ssim_scores.extend(ssim_fn(rec_gray[i], true_gray[i], data_range=1.0, gaussian_weights=True, sigma=1.5, use_sample_covariance=False) for i in range(n))

        rec_latents = encode_sdvae_images(rec)
        true_latents = cache["latents"].detach().cpu()
        latent_mse_total += n * float(F.mse_loss(rec_latents, true_latents).cpu())
        latent_cos_total += n * float(F.cosine_similarity(rec_latents.flatten(1), true_latents.flatten(1), dim=1).mean().cpu())
        seen += n
    return {
        "N": seen,
        "Pixel_MSE": pixel_mse_total / seen,
        "PixCorr": float(np.mean(pixcorr_scores)),
        "SSIM": float(np.mean(ssim_scores)),
        "SDVAE_Latent_MSE": latent_mse_total / seen,
        "SDVAE_Latent_Cosine": latent_cos_total / seen,
    }

metrics = eval_reconstruction_metrics(recons_eval, ids_eval)
metrics_df = pd.DataFrame([metrics]).set_index("N")
display(metrics_df)
metrics_df.to_csv(f"{OUTPUT_DIR}/mindeye_lowlevel_sdvae_shared1000_eval.csv")
print(f"saved={OUTPUT_DIR}/mindeye_lowlevel_sdvae_shared1000_eval.csv")


## Preview Grid


In [ ]:
PREVIEW_GRID_N = 48
PREVIEW_GRID_COLS = 6
PREVIEW_GRID_PATH = f"{OUTPUT_DIR}/mindeye_lowlevel_sdvae_preview_grid.png"

n_show = min(PREVIEW_GRID_N, len(all_recons))
rng = np.random.RandomState(SEED)
show_idx = rng.choice(len(all_recons), size=n_show, replace=False)
cols = min(PREVIEW_GRID_COLS, n_show)
rows = int(np.ceil(n_show / cols))

true_images = sdvae_targets_for_ids(torch.tensor([shared_img_ids[int(i)] for i in show_idx]), include_images=True)["images"].float()
pred_images = all_recons[show_idx].float()
combined = torch.cat([true_images, pred_images], dim=3).clamp(0, 1)

fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 3.0 * rows), constrained_layout=True)
axes = np.atleast_1d(axes).reshape(rows, cols)
for ax in axes.ravel():
    ax.axis("off")
for k, idx in enumerate(show_idx):
    ax = axes[k // cols, k % cols]
    ax.imshow(combined[k].permute(1, 2, 0))
    ax.set_title(f"shared id {shared_img_ids[int(idx)]}\nactual | predicted", fontsize=8)
    ax.axis("off")
plt.savefig(PREVIEW_GRID_PATH, dpi=180, bbox_inches="tight")
plt.show()
print(f"saved={PREVIEW_GRID_PATH}")
